# 01. Carga, comprensión e integración

Este notebook carga los dos archivos sin modificar los originales, documenta su granularidad y
sus llaves, explica las entidades y realiza una integración muchos-a-uno por `video_id`.

> Reproducibilidad: colocar `youtube_videos.csv` y `youtube_comments.csv` junto al notebook o
> dentro de una carpeta `data/`.

## 1.1 Carga de datos

Los identificadores se leen como texto desde el inicio para evitar conversiones accidentales y
preserva exactamente los valores usados como llaves.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Busca los CSV sin depender de una ruta absoluta del equipo
def localizar_archivo(nombre):
    candidatos = [
        Path.cwd() / nombre,
        Path.cwd() / "data" / nombre,
        Path.cwd().parent / "data" / nombre,
    ]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    buscadas = "\n- ".join(str(p) for p in candidatos)
    raise FileNotFoundError(
        f"No se encontró {nombre}. Colóquelo junto al notebook o en data/. "
        f"Rutas revisadas:\n- {buscadas}"
    )

RUTA_VIDEOS = localizar_archivo("youtube_videos.csv")
RUTA_COMENTARIOS = localizar_archivo("youtube_comments.csv")

ID_VIDEOS = {"video_id": "string", "channel_id": "string"}
ID_COMENTARIOS = {
    "video_id": "string",
    "comment_id": "string",
    "channel_id": "string",
    "author_channel_id": "string",
}

videos = pd.read_csv(RUTA_VIDEOS, dtype=ID_VIDEOS)
comentarios = pd.read_csv(RUTA_COMENTARIOS, dtype=ID_COMENTARIOS)

print(f"Videos: {videos.shape[0]:,} filas x {videos.shape[1]} columnas")
print(f"Comentarios: {comentarios.shape[0]:,} filas x {comentarios.shape[1]} columnas")
print(f"Archivos: {RUTA_VIDEOS.name}, {RUTA_COMENTARIOS.name}")


Videos: 293 filas x 20 columnas
Comentarios: 406 filas x 17 columnas
Archivos: youtube_videos.csv, youtube_comments.csv


In [2]:
resumen_carga = pd.DataFrame({
    "archivo": [RUTA_VIDEOS.name, RUTA_COMENTARIOS.name],
    "filas": [len(videos), len(comentarios)],
    "columnas": [videos.shape[1], comentarios.shape[1]],
    "memoria_KiB": [videos.memory_usage(deep=True).sum()/1024,
                     comentarios.memory_usage(deep=True).sum()/1024],
})
resumen_carga


,archivo,filas,columnas,memoria_KiB
0,youtube_videos.csv,293,20,348.615
1,youtube_comments.csv,406,17,222.041


In [3]:
print("Muestra de videos")
print(videos[["video_id", "title", "channel_name", "category", "view_count"]].head(3).to_string(index=False))
print("\nMuestra de comentarios")
print(comentarios[["comment_id", "video_id", "author_name", "text"]].head(3).to_string(index=False))


Muestra de videos
   video_id                                                                                         title           channel_name        category  view_count
-5puKGEqcUc                 INSIVUMEH pronostica incremento de lluvias para el fin de semana en Guatemala T13 Noticias Guatemala News & Politics        2357
-E7OPOLjMug BERNARDO ARÉVALO CALIFICA CAMBIO EN EL MP COMO EL FIN DE UNA ETAPA DE DETERIORO INSTITUCIONAL             IDocumenta  People & Blogs           4
-KDglrIzRKo                                ¡HISTÓRICO! Mexico recupera petróleo robado por Guatemala... 🔔           México Poder  People & Blogs       29736

Muestra de comentarios
                comment_id    video_id        author_name                                                                                                                                                                            text
Ugw-J65a1iYL9hqhELh4AaABAg j43HgwYFKfk @MarcosCarillo-b1r                                       

## 1.2 Unidad de observación, llave primaria y variables relevantes

| Archivo | Unidad de observación | Llave primaria | Variables relevantes |
|---|---|---|---|
| `youtube_videos.csv` | Un video publicado en YouTube | `video_id` | `channel_id`, `channel_name`, `title`, `category`, `source_query`, `query_hits`, `keywords`, `description`, `view_count`, `publish_date` |
| `youtube_comments.csv` | Un comentario principal observado en un video | `comment_id` | `video_id`, `author_channel_id`, `author_name`, `text`, `like_count_text`, `reply_count`, `source_query` |

Los nombres y handles sirven como etiquetas legibles, no como llaves. Un nombre puede cambiar y,
en otros conjuntos, podría repetirse. Los IDs estables se conservan como identificadores.


In [4]:
def revisar_llave(df, llave, archivo):
    return {
        "archivo": archivo,
        "llave": llave,
        "filas": len(df),
        "faltantes_llave": int(df[llave].isna().sum()),
        "valores_unicos": int(df[llave].nunique(dropna=True)),
        "duplicados_llave": int(df[llave].duplicated(keep=False).sum()),
        "es_llave_primaria": bool(df[llave].notna().all() and df[llave].is_unique),
    }

chequeo_llaves = pd.DataFrame([
    revisar_llave(videos, "video_id", "youtube_videos.csv"),
    revisar_llave(comentarios, "comment_id", "youtube_comments.csv"),
])
chequeo_llaves


,archivo,llave,filas,faltantes_llave,valores_unicos,duplicados_llave,es_llave_primaria
0,youtube_videos.csv,video_id,293,0,293,0,True
1,youtube_comments.csv,comment_id,406,0,406,0,True


In [5]:
tipos_videos = pd.DataFrame({
    "variable": videos.columns,
    "dtype_observado": videos.dtypes.astype(str).values,
    "no_nulos": videos.notna().sum().values,
    "unicos": videos.nunique(dropna=True).values,
})
tipos_comentarios = pd.DataFrame({
    "variable": comentarios.columns,
    "dtype_observado": comentarios.dtypes.astype(str).values,
    "no_nulos": comentarios.notna().sum().values,
    "unicos": comentarios.nunique(dropna=True).values,
})
print("Esquema observado: videos")
print(tipos_videos.to_string(index=False))
print("\nEsquema observado: comentarios")
print(tipos_comentarios.to_string(index=False))


Esquema observado: videos
           variable dtype_observado  no_nulos  unicos
           video_id          string       293     293
              title             str       293     274
       channel_name             str       293      97
         channel_id          string       293      97
       source_query             str       293      21
       source_group             str       293       3
    dataset_sources             str       293      23
     channel_handle             str       293      97
     published_time             str       280      80
    view_count_text             str       280     259
description_snippet             str       268     236
          video_url             str       293     293
         query_hits             str       293      26
           keywords             str       293     106
        description             str       267     234
         view_count           int64       293     267
       publish_date             str       293     293
  

## 1.3 Relación entre las entidades

- Un **canal** (`channel_id`) publica cero o más **videos** (`video_id`).
- Un **video** pertenece a un canal, tiene una **categoría** asignada por YouTube y pudo ser
  recuperado mediante una o más **consultas de búsqueda**. La consulta describe el muestreo; no
  equivale necesariamente al tema definitivo.
- Un **autor de comentario** (`author_channel_id`) puede publicar uno o más **comentarios**.
- Cada fila de comentarios observada pertenece a un solo video mediante `video_id`; por esa vía se
  obtiene el canal, la categoría y demás atributos del video.
- `channel_id` en comentarios identifica al canal propietario del video; `author_channel_id`
  identifica al autor. No deben confundirse.
- `reply_count` solo cuenta respuestas recibidas. No identifica quién respondió a quién y no crea
  una relación autor-autor.

Cardinalidades conceptuales: `canal 1:N video`, `video 1:N comentario` y
`autor 1:N comentario`. Autor y video forman una relación N:M al agrupar comentarios.


## 1.4 Integración mediante `video_id`

Se usa `validate="many_to_one"`: muchas filas de comentarios pueden corresponder a un video, pero
el archivo de videos debe contener a lo sumo una fila por `video_id`. `indicator=True` permite
auditar el resultado de la unión.


In [6]:
comentarios_integrados = comentarios.merge(
    videos,
    on="video_id",
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("_comentario", "_video"),
)

integracion = comentarios_integrados["_merge"].value_counts(dropna=False).rename_axis("resultado").reset_index(name="comentarios")
integracion["porcentaje"] = 100 * integracion["comentarios"] / len(comentarios_integrados)
integracion


,resultado,comentarios,porcentaje
0,both,406,100.000
1,left_only,0,0.000
2,right_only,0,0.000


In [7]:
asociados = int((comentarios_integrados["_merge"] == "both").sum())
no_asociados = int((comentarios_integrados["_merge"] == "left_only").sum())
videos_con_comentarios = comentarios["video_id"].nunique()

redundancias = pd.DataFrame({
    "comparación": ["video_title vs title", "channel_id", "channel_name"],
    "coincidencias": [
        (comentarios_integrados["video_title"].fillna("") == comentarios_integrados["title"].fillna("")).sum(),
        (comentarios_integrados["channel_id_comentario"].fillna("") == comentarios_integrados["channel_id_video"].fillna("")).sum(),
        (comentarios_integrados["channel_name_comentario"].fillna("") == comentarios_integrados["channel_name_video"].fillna("")).sum(),
    ],
})
redundancias["total"] = len(comentarios_integrados)

print(f"Comentarios asociados: {asociados:,} de {len(comentarios):,} ({asociados/len(comentarios):.1%})")
print(f"Comentarios sin video: {no_asociados:,}")
print(f"Videos con al menos un comentario observado: {videos_con_comentarios:,} de {len(videos):,}")
print("\nConsistencia de variables redundantes:")
print(redundancias.to_string(index=False))


Comentarios asociados: 406 de 406 (100.0%)
Comentarios sin video: 0
Videos con al menos un comentario observado: 19 de 293

Consistencia de variables redundantes:
         comparación  coincidencias  total
video_title vs title            406    406
          channel_id            406    406
        channel_name            406    406


In [8]:
columnas_revision = [
    "comment_id", "video_id", "author_channel_id", "author_name", "text",
    "title", "channel_id_video", "channel_name_video", "category", "view_count"
]
comentarios_integrados[columnas_revision].head(5)


,comment_id,video_id,author_channel_id,author_name,text,title,channel_id_video,channel_name_video,category,view_count
0,Ugw-J65a1iYL9hqhELh4AaABAg,j43HgwYFKfk,UCdFlugHJJa4l3YqWuNRmvXw,@MarcosCarillo-b1r,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,La cooptación de Walter Mazariegos en la USAC,UCE4rsXcgDb6e1-a9iTbWzfg,Quorum,News & Politics,10156
1,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,06mFNPU0aB8,UCvl1tzQeBeGy6efPTRJXSCw,@RaulPerez-cw2vi,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay policías que le...",Capturan a presuntos delincuentes disfrazados de mujer señalados de cometer asalto,UCVpSRoZgngfSL03Nlbjtq9A,Noti7,News & Politics,6692
2,Ugw0xaOb2CYXXoudtwJ4AaABAg,j43HgwYFKfk,UCRAquv8el-tQ30bN7MlmySQ,@iamjimalesssa,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, esto no es ...",La cooptación de Walter Mazariegos en la USAC,UCE4rsXcgDb6e1-a9iTbWzfg,Quorum,News & Politics,10156
3,Ugw0xgUc2ISpBr5_T654AaABAg,j43HgwYFKfk,UCkbsS_3D-pvg1iHOld9uvIg,@gonzalo6075,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la sombra.,La cooptación de Walter Mazariegos en la USAC,UCE4rsXcgDb6e1-a9iTbWzfg,Quorum,News & Politics,10156
4,Ugw1ZzA21njWhaqQQTh4AaABAg,OkXlHx0hx-8,UCOLHH4ZpMxPYF-Q6e6wvn5A,@luisroldan7374,eso es para que salga de USA por su propio pie \nque se auto deporten,EE.UU. envía a mexicanos deportados a Guatemala antes de su regreso a México | Noticia...,UCRwA1NUcUnwsly35ikGhp0A,Noticias Telemundo,News & Politics,14200


In [9]:
assert videos["video_id"].is_unique and videos["video_id"].notna().all()
assert comentarios["comment_id"].is_unique and comentarios["comment_id"].notna().all()
assert no_asociados == 0
assert len(comentarios_integrados) == len(comentarios)
print("Validaciones superadas: llaves únicas, cardinalidad many-to-one y 100% de asociación.")


Validaciones superadas: llaves únicas, cardinalidad many-to-one y 100% de asociación.


## Conclusión del punto 1

Los 406 comentarios se asociaron con un registro de video (100%). Sin embargo, solo 19 de los
293 videos tienen comentarios en este archivo. Esta diferencia describe la cobertura de la
recolección y no demuestra que los otros videos carezcan de comentarios en YouTube.
